# 第 11 章：多 Agent 模式——Router、Handoff、Supervisor 与 Subagent-as-tool（离线工程实验）

**目标**：执行本章稳定公共契约并观察失败护栏。  
**环境与预计用时**：Python 3.12、offline profile，约 15–25 分钟。  
本 Notebook 由同名 Markdown 中带 `sync` 标识的实验代码生成；可复用业务逻辑始终从 `mini_deerflow` package 导入。

## 1. offline profile 初始化

显式选择离线模型档位，基础实验不得读取供应商 Key。

In [1]:
from mini_deerflow.config import ModelProfile, ModelSettings

lesson_settings = ModelSettings(profile=ModelProfile.OFFLINE)
assert lesson_settings.profile is ModelProfile.OFFLINE


## 2. 前置能力探针

验证当前 kernel 使用课程锁定的主版本，并能导入 Mini DeerFlow。

In [2]:
from importlib.metadata import version
import mini_deerflow

assert version('langchain').startswith('1.3.')
assert version('langgraph').startswith('1.2.')
assert mini_deerflow.__file__


## 3. 最小成功实验

以下单元来自 Markdown 的稳定 sync marker。

### 实验 `ch11-command-router`

In [3]:
from mini_deerflow.subagents import build_single_router_graph

single_router = build_single_router_graph()
single_route_result = single_router.invoke({"query": "研究 LangGraph checkpoint"})

assert single_route_result["route"] == "research"
assert single_route_result["trace"] == ["router:research", "research"]
assert single_route_result["answer"] == "research specialist result"


### 实验 `ch11-send-router`

In [4]:
from mini_deerflow.subagents import build_parallel_router_graph

parallel_router = build_parallel_router_graph()
parallel_route_result = parallel_router.invoke(
    {
        "query": "比较研究结论与 Python 实现",
        "routes": ["research", "coding"],
    }
)

assert sorted(item.agent_name for item in parallel_route_result["results"]) == [
    "coding",
    "research",
]
assert parallel_route_result["answer"] == "coding+research"


### 实验 `ch11-handoff`

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from mini_deerflow.subagents import build_handoff_graph

handoff_graph = build_handoff_graph(checkpointer=InMemorySaver())
handoff_config = {"configurable": {"thread_id": "handoff-lesson"}}
first_handoff = handoff_graph.invoke(
    {"request": "请修复 Python 类型错误"},
    config=handoff_config,
)
handoff_result = handoff_graph.invoke(
    {"request": "继续解释这个修复"},
    config=handoff_config,
)

assert first_handoff["active_agent"] == "coding"
assert handoff_result["active_agent"] == "coding"
assert handoff_result["trace"] == [
    "triage->coding",
    "coding:answered",
    "coding:answered",
]


### 实验 `ch11-isolated-specialists`

In [6]:
import asyncio
from mini_deerflow.subagents import (
    SubagentExecutor,
    SubagentRequest,
    build_demo_subagent_registry,
)

demo_registry = build_demo_subagent_registry()
demo_executor = SubagentExecutor(demo_registry, max_concurrency=2)

async def run_demo_specialists():
    return await demo_executor.dispatch_many(
        [
            SubagentRequest(
                task_id="lesson-research",
                agent_name="research",
                description="研究 reducer",
                prompt="解释并行 reducer 的边界",
            ),
            SubagentRequest(
                task_id="lesson-coding",
                agent_name="coding",
                description="设计 reducer 测试",
                prompt="给出防止重复合并的测试建议",
            ),
        ],
        parent_context={
            "locale": "zh-CN",
            "messages": ["完整主会话不应进入 specialist"],
            "auth_token": "demo-secret-must-not-leak",
        },
    )

demo_results = asyncio.run(run_demo_specialists())
assert [result.status for result in demo_results] == ["completed", "completed"]
assert demo_results[0].summary.startswith("研究摘要")
assert demo_results[1].summary.startswith("代码建议")
assert all("demo-secret" not in result.model_dump_json() for result in demo_results)


### 实验 `ch11-task-tool`

In [7]:
import json
from mini_deerflow.schemas import SubagentResult
from mini_deerflow.subagents import build_task_tool

task_tool = build_task_tool(
    demo_executor,
)
assert task_tool.metadata["max_concurrency"] == 2
assert "runtime" not in task_tool.args


### 实验 `ch11-lead-agent-supervisor`

In [8]:
from langchain_core.messages import AIMessage, ToolMessage
from mini_deerflow.agents import create_lead_agent
from mini_deerflow.config import LeadAgentContext
from mini_deerflow.models import create_offline_model

supervisor_model = create_offline_model(
    [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "task",
                    "args": {
                        "task_id": "lead-delegation-001",
                        "description": "研究 checkpoint",
                        "prompt": "只返回三条恢复原则",
                        "subagent_type": "research",
                    },
                    "id": "lead-tool-call-001",
                    "type": "tool_call",
                }
            ],
        ),
        AIMessage(content="Lead Agent 已审阅并汇总 specialist 结果。"),
    ]
)
supervisor = create_lead_agent(model=supervisor_model, tools=[task_tool])
supervisor_state = asyncio.run(
    supervisor.ainvoke(
        {"messages": [{"role": "user", "content": "解释 checkpoint 恢复"}]},
        context=LeadAgentContext(
            user_id="lesson-user",
            workspace_root="/tmp/lesson-workspace",
            auth_token="never-forward",
        ),
    )
)

delegation_message = next(
    message for message in supervisor_state["messages"] if isinstance(message, ToolMessage)
)
delegation_result = SubagentResult.model_validate_json(delegation_message.content)
assert delegation_result.status == "completed"
assert delegation_result.agent_name == "research"
assert supervisor_state["messages"][-1].content.startswith("Lead Agent 已审阅")


### 实验 `ch11-concurrency-limit`

In [9]:
from mini_deerflow.subagents import (
    SubagentInvocation,
    SubagentOutput,
    SubagentRegistry,
    SubagentSpec,
)

concurrency = {"active": 0, "peak": 0}

async def measured_worker(invocation: SubagentInvocation) -> SubagentOutput:
    concurrency["active"] += 1
    concurrency["peak"] = max(concurrency["peak"], concurrency["active"])
    await asyncio.sleep(0.01)
    concurrency["active"] -= 1
    return SubagentOutput(summary=f"done:{invocation.prompt}")

measured_executor = SubagentExecutor(
    SubagentRegistry(
        [SubagentSpec(name="worker", description="测量并发", handler=measured_worker)]
    ),
    max_concurrency=2,
)
measured_requests = [
    SubagentRequest(
        task_id=f"concurrency-{index}",
        agent_name="worker",
        description="并发实验",
        prompt=str(index),
    )
    for index in range(4)
]
measured_results = asyncio.run(measured_executor.dispatch_many(measured_requests))

assert concurrency["peak"] == 2
assert all(result.status == "completed" for result in measured_results)


## 4. 状态/事件观察

观察消息、结构化对象、检索命中或 v2 event；不要只看最终自然语言。

### 实验 `ch11-delegation-ledger-event`

In [10]:
ledger_records = demo_executor.ledger.list_records()

assert len(ledger_records) >= 3
latest_record = ledger_records[-1]
assert latest_record.status == "completed"
assert latest_record.context_keys == ("locale", "request_id")
assert len(latest_record.output_sha256 or "") == 64
assert "never-forward" not in latest_record.model_dump_json()
print(latest_record.model_dump())


{'task_id': 'lead-delegation-001', 'agent_name': 'research', 'status': 'completed', 'context_keys': ('locale', 'request_id'), 'summary_preview': '研究摘要[zh-CN]：只返回三条恢复原则', 'output_chars': 21, 'output_sha256': 'e091147c45f175752a6506803ad7b74a7ba758721da3c077fd4198c56f2c195d', 'error_code': None}


## 5. 失败实验

失败必须被捕获并断言，证明护栏真的阻止了错误路径。

### 实验 `ch11-subgraph-boundary`

In [11]:
from mini_deerflow.subagents import build_shared_subgraph_graph

shared_subgraph = build_shared_subgraph_graph()
subgraph_result = shared_subgraph.invoke(
    {"query": "解释 reducer", "notes": ["parent"]}
)

assert subgraph_result["notes"] == ["parent", "subgraph:解释 reducer"]


### 实验 `ch11-context-pollution-failure`

In [12]:
from mini_deerflow.subagents import (
    SubagentInvocation,
    SubagentOutput,
    SubagentRegistry,
    SubagentSpec,
)

polluted_parent_context = {
    "user_id": "learner-11",
    "locale": "zh-CN",
    "messages": ["无关的长主会话"],
    "internal_notes": "Lead 私有草稿",
    "auth_token": "leaked-by-unsafe-copy",
}
unsafe_child_context = dict(polluted_parent_context)
assert "messages" in unsafe_child_context
assert unsafe_child_context["auth_token"] == "leaked-by-unsafe-copy"

observed_contexts = []

async def inspect_context(invocation: SubagentInvocation) -> SubagentOutput:
    observed_contexts.append(invocation.context)
    return SubagentOutput(summary="context inspected")

for secret_alias in (
    "client_secret",
    "openai_api_key",
    "auth_token",
    "access_token",
    "refresh_token",
):
    try:
        SubagentSpec(
            name="unsafe",
            description="错误输入策略",
            handler=inspect_context,
            allowed_context_fields=frozenset({secret_alias}),
        )
    except ValueError as secret_policy_error:
        assert "secret 字段" in str(secret_policy_error)
    else:
        raise AssertionError(f"{secret_alias} 不应进入 allowlist")

isolated_registry = SubagentRegistry(
    [
        SubagentSpec(
            name="inspector",
            description="检查输入边界",
            handler=inspect_context,
            allowed_context_fields=frozenset({"user_id", "locale"}),
        )
    ]
)
isolated_executor = SubagentExecutor(isolated_registry)
isolated_result = asyncio.run(
    isolated_executor.dispatch(
        SubagentRequest(
            task_id="context-001",
            agent_name="inspector",
            description="验证输入边界",
            prompt="只报告允许字段",
        ),
        parent_context=polluted_parent_context,
    )
)

assert isolated_result.status == "completed"
assert observed_contexts == [{"user_id": "learner-11", "locale": "zh-CN"}]
assert "leaked-by-unsafe-copy" not in isolated_result.model_dump_json()


### 实验 `ch11-timeout-partial-failure`

In [13]:
async def sometimes_fails(invocation: SubagentInvocation) -> SubagentOutput:
    if invocation.prompt == "timeout":
        await asyncio.sleep(0.05)
    if invocation.prompt == "failure":
        raise RuntimeError("offline source unavailable")
    return SubagentOutput(summary=f"ok:{invocation.prompt}")

failure_executor = SubagentExecutor(
    SubagentRegistry(
        [
            SubagentSpec(
                name="unstable",
                description="故障注入 specialist",
                handler=sometimes_fails,
            )
        ]
    ),
    max_concurrency=2,
    timeout_seconds=0.01,
)
failure_results = asyncio.run(
    failure_executor.dispatch_many(
        [
            SubagentRequest(
                task_id=f"failure-{index}",
                agent_name="unstable",
                description="故障实验",
                prompt=value,
            )
            for index, value in enumerate(["success", "failure", "timeout"])
        ]
    )
)

assert [result.status for result in failure_results] == [
    "completed",
    "failed",
    "timed_out",
]
assert failure_results[1].error == "RuntimeError: subagent handler failed"
assert "执行预算" in (failure_results[2].error or "")


### 实验 `ch11-output-budget-failure`

In [14]:
from mini_deerflow.schemas import ArtifactRef

async def verbose_worker(_: SubagentInvocation) -> SubagentOutput:
    return SubagentOutput(
        summary="证据" * 80,
        artifacts=[
            ArtifactRef(path=f"reports/{index}.md", media_type="text/markdown")
            for index in range(5)
        ],
    )

bounded_executor = SubagentExecutor(
    SubagentRegistry(
        [
            SubagentSpec(
                name="verbose",
                description="产生大输出",
                handler=verbose_worker,
                max_output_chars=32,
                max_artifacts=2,
            )
        ]
    )
)
bounded_result = asyncio.run(
    bounded_executor.dispatch(
        SubagentRequest(
            task_id="large-output-001",
            agent_name="verbose",
            description="验证输出预算",
            prompt="返回大量证据",
        )
    )
)

assert bounded_result.status == "output_too_large"
assert bounded_result.truncated is True
assert len(bounded_result.summary) == 32
assert bounded_result.output_chars == 160
assert len(bounded_result.output_sha256 or "") == 64
assert len(bounded_result.artifacts) == 2


## 6. Mini DeerFlow 工程调用

以上实验只从 `mini_deerflow` 导入公共接口；Notebook 不复制 Agent、Tool 或 Schema 实现。

## 7. 分层练习

完成同名 Markdown 的练习 A（单点修改）、B（边界判断）、C（项目扩展）和延迟回忆题。先自行作答，再运行对应 pytest 获取即时反馈。

## 8. 自动验收摘要

在项目根目录运行 `make test`。本 Notebook 的所有代码单元必须有执行计数、不得保存 error output，教程验证结果不得出现本章 drift。

## 9. 清理临时资源

当前实验使用内存对象与 `TemporaryDirectory`，退出上下文后自动清理；不要把 API Key、向量库或临时产物写回仓库。